# Module 01 — En terrain connu

**Formation Big Data — ANSD / Data Innovation Lab**

Dans ce premier notebook : on travaille sur un **échantillon** du
recensement avec les outils habituels, pour installer nos repères.

Objectifs :

1. découvrir le jeu de données qui nous accompagnera toute la semaine ;
2. réaliser quelques calculs statistiques courants avec pandas ;
3. **chronométrer chaque opération** — ces temps seront notre point de
   comparaison dans les notebooks suivants.

> Les cellules marquées **`# À COMPLÉTER`** sont à remplir. Tout le reste est
> fourni : ne perdez pas de temps sur la "plomberie".

## 1. Préparation

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd

DOSSIER_DONNEES = Path("..") / "00-data"
FICHIER = DOSSIER_DONNEES / "individus.csv"
FICHIER_REGIONS = DOSSIER_DONNEES / "regions.csv"

# Journal des mesures : chaque opération chronométrée y sera enregistrée.
mesures = []


def chrono(libelle, fonction, taille=None):
    """Exécute `fonction`, affiche et enregistre son temps d'exécution."""
    depart = time.perf_counter()
    resultat = fonction()
    duree = time.perf_counter() - depart
    mesures.append({"operation": libelle, "lignes": taille, "secondes": round(duree, 3)})
    print(f"{libelle:<28} {duree:>8.3f} s")
    return resultat


print("pandas", pd.__version__, "| numpy", np.__version__)
print("Fichier présent :", FICHIER.exists())

### Chargement d'un échantillon

On ne lit que les **100 000 premières lignes** : de quoi travailler
confortablement, sans solliciter la machine.

In [ ]:
N_ECHANTILLON = 100_000

df = chrono(
    "Lecture 100k lignes",
    lambda: pd.read_csv(FICHIER, nrows=N_ECHANTILLON),
    taille=N_ECHANTILLON,
)

df.shape

## 2. Découverte du jeu de données

Un fichier de recensement simulé : un individu par ligne, regroupé en ménages.
Prenez deux minutes pour regarder ce qu'il contient avant de calculer quoi que
ce soit — c'est un réflexe de statisticien qui reste valable à toutes les
échelles.

In [ ]:
df.head(10)

In [ ]:
df.dtypes

In [ ]:
# Mémoire occupée par le DataFrame, colonne par colonne (en Mo)
(df.memory_usage(deep=True) / 1024**2).round(2).sort_values(ascending=False)

**Question 1.** Le fichier complet contient plusieurs millions de lignes.
D'après la mémoire occupée par ces 100 000 lignes, combien de mémoire faudrait-il
pour tout charger ? Notez votre estimation, nous la vérifierons au notebook
suivant.

*Votre estimation :* … Go

### Qualité des données

Ces données reproduisent les défauts habituels des fichiers administratifs.
Repérons-les.

In [ ]:
# À COMPLÉTER — part de valeurs manquantes par colonne, en %,
# en ne gardant que les colonnes concernées, triées par ordre décroissant.
# Indice : .isna(), .mean(), * 100

manquants = ...
manquants

In [ ]:
# À COMPLÉTER — la colonne `region` contient-elle des libellés incohérents ?
# Affichez les valeurs distinctes et comptez-les.

...

In [ ]:
# Fourni : distribution des âges — repérez les valeurs impossibles
df["age"].describe()

**Question 2.** Trois anomalies au moins sont visibles dans les cellules
ci-dessus. Lesquelles, et quelle serait votre stratégie de traitement pour
chacune ?

*Votre réponse :*
1. …
2. …
3. …

## 3. Trois calculs métier

Rien de nouveau ici : ce sont les tableaux que vous produisez habituellement.
L'objectif est de **mesurer** combien de temps ils prennent sur 100 000 lignes.

In [ ]:
# À COMPLÉTER — effectifs par région, du plus peuplé au moins peuplé.
# Attention aux libellés incohérents repérés plus haut : nettoyez-les d'abord
# (indice : .str.strip().str.title()).

def effectifs_par_region():
    ...

effectifs = chrono("Effectifs par région", effectifs_par_region, taille=N_ECHANTILLON)
effectifs

In [ ]:
# À COMPLÉTER — âge moyen par sexe, en excluant les âges aberrants.

def age_moyen_par_sexe():
    ...

chrono("Âge moyen par sexe", age_moyen_par_sexe, taille=N_ECHANTILLON)

In [ ]:
# À COMPLÉTER — taux d'activité par milieu de résidence, chez les 15 ans et plus.
# Définition retenue : part des personnes « Occupé » ou « Chômeur »
# parmi les 15 ans et plus.

def taux_activite():
    ...

chrono("Taux d'activité", taux_activite, taille=N_ECHANTILLON)

## 4. Une jointure avec une table de référence

La table `regions.csv` porte le code région, le chef-lieu et la superficie.
On l'associe au fichier individus pour calculer une densité par région.

In [ ]:
regions = pd.read_csv(FICHIER_REGIONS)
regions

In [ ]:
# À COMPLÉTER — joindre `regions` au DataFrame nettoyé, puis calculer
# le nombre d'individus de l'échantillon par km² et par région.

def jointure_regions():
    ...

densite = chrono("Jointure + densité", jointure_regions, taille=N_ECHANTILLON)
densite

## 5. Le tableau de mesures

Voici notre point de comparaison pour la suite de la journée. Conservez-le :
nous le reprendrons avec Dask, puis avec Parquet.

In [ ]:
tableau = pd.DataFrame(mesures)
tableau["lignes_par_seconde"] = (tableau["lignes"] / tableau["secondes"]).round(0)
tableau

In [ ]:
# Sauvegarde des mesures, pour comparaison dans les notebooks suivants
Path("resultats").mkdir(exist_ok=True)
tableau.to_csv("resultats/mesures_pandas_100k.csv", index=False)
print("Mesures enregistrées.")

## 6. Ce qu'il faut retenir

Sur 100 000 lignes, tout fonctionne : les temps se comptent en dixièmes de
seconde et la mémoire consommée reste modeste. C'est exactement l'usage pour
lequel pandas a été conçu.

**À compléter :**

- Temps total des opérations ci-dessus : … secondes
- Mémoire occupée par l'échantillon : … Mo
- Mon estimation pour le fichier complet : … Go

➡️ Notebook suivant : `02_le_mur_de_la_memoire.ipynb` — on charge le fichier
entier et on vérifie cette estimation.